# AWS SQS Streaming

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/aws_sqs/sqs_demo.ipynb) 
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-blue?logo=github)](https://github.com/LineageLogic/LakeLogic/blob/main/examples/03_data_sources/streaming/aws_sqs/sqs_demo.ipynb)

## Business Scenario

E-commerce and microservice systems publish events to SQS. You need a reliable way to ingest messages and validate them before storage.

## Value Proposition

- Reliable ingestion with validation at the edge
- Contract-based schema enforcement for events
- Quarantine invalid messages instead of losing them

---

## Goals

1. Connect to an SQS queue
2. Stream and validate events
3. Materialize clean records


##  Step 1: Setup AWS SQS

Before running this notebook, you need:
1. An AWS SQS queue (Standard or FIFO)
2. AWS credentials configured (via `~/.aws/credentials` or IAM role)
3. The queue URL

Set your environment variable:
```bash
export AWS_SQS_QUEUE_URL="https://sqs.us-east-1.amazonaws.com/123456789012/my-queue"
```

##  Step 2: Review the Contract

Our contract defines the expected message schema and quality rules.

In [ ]:
from pathlib import Path
from lakelogic import DataProcessor

BASE = Path.cwd()
contract_path = BASE / "sqs_contract.yaml"
if not contract_path.exists():
    candidate = BASE / "examples" / "03_data_sources" / "streaming" / "aws_sqs" / "sqs_contract.yaml"
    if candidate.exists():
        BASE = candidate.parent
        contract_path = candidate

RUN_LIVE = False  # Set True when your streaming infrastructure is available

sample_events = [{'orderId': 'ORD-001', 'customerId': 'CUST-123', 'amount': 99.99, 'currency': 'USD', 'timestamp': '2026-02-15T10:00:00', 'status': 'pending'}, {'orderId': 'ORD-002', 'customerId': 'CUST-456', 'amount': 149.5, 'currency': 'EUR', 'timestamp': '2026-02-15T10:05:00', 'status': 'completed'}]

if RUN_LIVE:
    print("Live streaming is disabled by default in this demo.")
    print("Set RUN_LIVE = True and configure credentials/endpoints to stream.")
else:
    processor = DataProcessor(contract=contract_path)
    result = processor.run(sample_events, source_path="sample")
    print(result)
    print(f"Good: {len(result.good)} | Bad: {len(result.bad)}")


##  Step 3: Start the SQS Consumer

This will connect to your SQS queue and start processing messages.

##  Step 4: Send Test Messages

Let's send some test order events to the queue.

##  Step 5: Verify Results

Check the materialized Delta table.

##  Summary

You just:
-  Connected to AWS SQS
-  Validated order events against a contract
-  Materialized events to Delta Lake

This pattern enables **reliable, scalable ingestion** from microservices architectures!